In [1]:
from kids.catalog.catalog import load_and_prepare_catalog
import pandas as pd

In [2]:
df = load_and_prepare_catalog('../data/KiDS_DR5_WCScut_columns_15MLN.csv')
df = df.merge(pd.read_csv('../data/inference/KiDS_DR5_WCScut_columns_SPECTYPE.csv', usecols=['ID', 'PREDICTED_CLASS']), how='inner', on='ID')

In [3]:
import numpy as np

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from kids.features import clf_features

In [4]:
preprocess_features = Pipeline([('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
                                ('scaler', RobustScaler())
                               ])

In [5]:
preprocess_features.fit(df.loc[:, clf_features].sample(2_500_000))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](91,)","['u-g','u-r','u-i1',...,'J/Ks','H/Ks','CLASS_STAR']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,91
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool

In [6]:
from joblib import dump, load
dump(preprocess_features, '../saved_models/som/preprocessor_robust_scaler.joblib')

['../saved_models/som/preprocessor_robust_scaler.joblib']

In [7]:
features = preprocess_features.transform(df.loc[:, clf_features])
# features = df.loc[:, clf_features]

In [ ]:
import torch
from kids.ml.som.som import MySOM

In [9]:
features = torch.tensor(features, dtype=torch.float32)

CLASS_TO_ID = {'QSO': 2, 'GALAXY': 1, 'STAR': 0}
targets = torch.tensor(df.loc[:, 'PREDICTED_CLASS'].map(CLASS_TO_ID), dtype=torch.long)

In [10]:
from dataclasses import asdict, dataclass

@dataclass
class SOMConfig:
    x: int = 60 
    y: int = 60 
    num_features: int = features.size(1) 
    epochs: int = 35
    batch_size: int = 4*1024 
    learning_rate: float = 0.8 
    sigma: int = max(x, y) // 2  
    device: str = 'cuda'
    topology: str = 'rectangular'
    pbc: bool = True
    lr_decay_function: str = 'asymptotic_decay' #'lr_inverse_decay_to_zero'
    sigma_decay_function: str = 'lr_linear_decay_to_zero' #'asymptotic_decay'
    neighborhood_function: str ='gaussian'
    distance_function: str ='euclidean'
    initialization_mode: str = 'pca'

config = asdict(SOMConfig())

In [11]:
som = MySOM(**config)
som.initialize_weights(data=features, mode=som.initialization_mode)
q_errors, t_errors = som.fit(data=features)

torch.cuda.empty_cache()

bmus_map = som.build_map('bmus_data', data=features, return_indices=True)

[W1030 12:36:56.259909059 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 3961520128 bytes (free: 3814129664, total: 12343050240).
/home/kacper-drabicki/Desktop/projects/KiDS-work/.venv/lib/python3.14/site-packages/torchsom/utils/initialization.py:163: UserWarning: PCA initialization failed: CUDA error: CUBLAS_STATUS_ALLOC_FAILED when calling `cublasCreate(handle)`. Falling back to random initialization
  return pca_init(weights, data, topology, device)


Training SOM:   0%|          | 0/35 [00:00<?, ?epoch/s]

RuntimeError: CUDA error: CUBLAS_STATUS_ALLOC_FAILED when calling `cublasCreate(handle)`

In [ ]:
torch.save(som.state_dict(), '../saved_models/som/test_som_60_60.pth')

In [ ]:
from torchsom.visualization import SOMVisualizer

In [ ]:
visualizer = SOMVisualizer(som=som)
visualizer.plot_all(
    quantization_errors=q_errors,
    topographic_errors=t_errors,
    bmus_data_map=bmus_map,
    data=features,
    target=targets,
    component_names=clf_features,
)

In [ ]:
visualizer.plot_classification_map(
    bmus_data_map=bmus_map,
    data=features,
    target=targets,
    neighborhood_order=som.neighborhood_order,
)